In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')

# [Step 1] 데이터 로드 시 최적화
dtypes = {
    'BUILDING_CD': 'category',
    'ROOM_CD': 'category',
    'READ_YMD': 'int32',
    'READ_HHMI': 'int16',
    'ELEC_INTER': 'float32'
}

print("1. 데이터 로드 및 타입 최적화 중")
df = pd.read_csv('홈에너지_시간별_계측.csv', dtype=dtypes) 

# [Step 2] 시간 변수 생성
df['READ_HHMI_str'] = df['READ_HHMI'].astype(str).str.zfill(4) 
df['datetime'] = pd.to_datetime(df['READ_YMD'].astype(str) + df['READ_HHMI_str'], format='%Y%m%d%H%M')

df['month'] = df['datetime'].dt.month.astype('int8')
df['day_of_week'] = df['datetime'].dt.dayofweek.astype('int8')
df['hour'] = df['datetime'].dt.hour.astype('int8')

# 데이터 정렬
df = df.sort_values(by=['BUILDING_CD', 'ROOM_CD', 'datetime']).reset_index(drop=True)

# [Step 3] 라벨 인코딩 (메모리 절약)
TARGET = 'ELEC_INTER'
le = LabelEncoder()
for col in ['BUILDING_CD', 'ROOM_CD']:
    df[col] = le.fit_transform(df[col])

FEATURES = [col for col in df.columns if col not in [TARGET, 'datetime', 'READ_YMD', 'READ_HHMI', 'READ_HHMI_str'] and 'ACCUM' not in col]

# [Step 4] 시간순 분할
split_index = int(len(df) * 0.8) 
train_df = df.iloc[:split_index] 
test_df = df.iloc[split_index:]  

# 여기서 X_train 생성 시 메모리 추가 할당 방지
X_train = train_df[FEATURES].values # .values를 사용하여 Numpy 배열로 변환 (메모리 효율)
y_train = train_df[TARGET].values
X_test = test_df[FEATURES].values
y_test = test_df[TARGET].values

# ==============================================================================
# [Step 4.5] 모든 변수를 강제로 숫자형(float)으로 변환
# ==============================================================================
# 문자열 데이터가 남아있으면 오류 발생함
for col in FEATURES:
    if train_df[col].dtype == 'object' or train_df[col].dtype.name == 'category':
        # 재확인
        train_df[col] = pd.to_numeric(train_df[col], errors='coerce')
        test_df[col] = pd.to_numeric(test_df[col], errors='coerce')

# 결측치(숫자 변환 실패 등)가 있다면 0으로 채움
X_train = train_df[FEATURES].fillna(0).values
X_test = test_df[FEATURES].fillna(0).values
y_train = train_df[TARGET].values
y_test = test_df[TARGET].values

# [Step 5] 학습 및 검증
print("4. Random Forest 모델 학습 시작")
model = RandomForestRegressor(n_estimators=50, max_depth=8, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

preds = np.maximum(model.predict(X_test), 0)
print(f"평균 절대 오차 (MAE): {mean_absolute_error(y_test, preds):.4f} kWh")